# QF600 Asset Pricing — Backtesting a Portfolio Strategy & Computing Alpha

Backtest an **investor portfolio** against a **benchmark portfolio**, both rebalanced on a fixed
schedule, then decompose the result into the part explained by market exposure and the part that is not:

$$R_p - R_f \;=\; \alpha \;+\; \beta\,(R_m - R_f) \;+\; \varepsilon$$


Documentation in this [Github link](https://github.com/cyee02/MQF/tree/main/QF600/Assignment%201%20-%20Compute%20Alpha)

---
## 1. Set up

All third-party libraries are declared here and nowhere else in the notebook.

In [7]:
# Environment bootstrap: install anything missing, then switch on an interactive table viewer.
import importlib
import subprocess
import sys


def ensure_installed(package: str, module: str | None = None) -> None:
    """pip-install `package` only if `module` cannot already be imported.

    Parameters
    ----------
    package : str         name to pass to `pip install`
    module  : str | None  import name, when it differs from `package` (e.g. "lets_plot"
                          for "lets-plot"); defaults to `package`

    Returns
    -------
    None  called for its side effect: the package is importable afterwards
    """
    try:
        importlib.import_module(module or package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


ensure_installed("yfinance")
ensure_installed("lets-plot", "lets_plot")
ensure_installed("tabulate")            # pandas needs it for DataFrame.to_markdown() in §8

try:
    import google.colab                                              # noqa: F401
    IN_COLAB = True
    get_ipython().run_line_magic("load_ext", "google.colab.data_table")  # paginated tables in Colab
except ImportError:
    IN_COLAB = False                                                 # local -> Data Wrangler plugin

print(f"Running in Colab: {IN_COLAB}")

Running in Colab: False


In [8]:
import datetime as dt

import numpy as np
import pandas as pd
import yfinance as yf
from scipy import stats

from IPython.display import Markdown, display
from lets_plot import *
LetsPlot.setup_html()

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.6f}".format)
pd.set_option("display.max_rows", 10)

---
## 2. Parameters

Everything a user would want to change lives in this one cell.

The strategy horizon is expressed as a **start / end date** rather than a raw day count: the spec
fixes it at "01 Jan 2016", and pinning the dates avoids the calendar-days vs trading-days ambiguity.
The horizon *in trading days* is derived from the data further below.

In [9]:
INVESTOR_PORTFOLIO  = {"SOXX": 0.70,          # semiconductor ETF
                       "GLD" : 0.30}          # gold ETF
# INVESTOR_PORTFOLIO  = {"XLK": 0.50,
#                        "QQQ" : 0.50}

BENCHMARK_PORTFOLIO = {"IVV" : 0.60,          # S&P 500 ETF
                       "AGG" : 0.40}          # US aggregate bond ETF

# START_DATE       = "2016-01-01"               # ~10 years of history
START_DATE       = "2014-01-01"
END_DATE         = "2018-12-31"
END_DATE         = dt.date.today()
REBALANCE_DAYS   = 60                         # trading days between rebalances (~1 quarter)
# REBALANCE_DAYS = int((END_DATE - dt.date.fromisoformat(START_DATE)).days * 0.7)  # No rebalance for now, just compute the returns of the portfolio over the entire period
INITIAL_CAPITAL  = 100_000                    # USD at inception
TRADING_DAYS     = 252                        # annualisation factor
CVAR_CONFIDENCE  = 0.95                       # CVaR tail cutoff: average of the worst 5% of days
# RISK_FREE_TICKER = "^TNX"                     # CBOE 10-year Treasury yield index
RISK_FREE_TICKER = "^FVX"                     # CBOE 5-year Treasury yield index

for name, portfolio in [("Investor", INVESTOR_PORTFOLIO), ("Benchmark", BENCHMARK_PORTFOLIO)]:
    assert np.isclose(sum(portfolio.values()), 1.0), f"{name} weights must sum to 1.0"

TICKERS = list(dict.fromkeys([*INVESTOR_PORTFOLIO, *BENCHMARK_PORTFOLIO]))   # de-duplicated
TICKERS

['SOXX', 'GLD', 'IVV', 'AGG']

---
## 3. Data

### 3.1 Daily close prices

The download is kept twice: `prices_raw` exactly as returned, so §3.2 can measure what is
missing, and `prices` restricted to dates every ticker traded, which is what the backtest uses.

In [10]:
prices_raw =\
(
    yf.download(TICKERS,                # every ticker in one request
                start        = START_DATE,
                end          = END_DATE,
                auto_adjust  = True,    # adjust for splits & dividends -> total-return prices
                progress     = False
               )
    ["Close"]                           # daily close only
    [TICKERS]                           # restore the declared column order
)

prices =\
(
    prices_raw
    .dropna()                           # keep only dates every ticker traded
)

prices

Ticker,SOXX,GLD,IVV,AGG
Date,,,,
2014-01-02,20.926954,118.000000,148.633163,75.132599
2014-01-03,20.865675,119.290001,148.568512,75.160820
2014-01-06,20.763533,119.500000,148.164642,75.259613
2014-01-07,20.906523,118.820000,149.077393,75.294876
2014-01-08,21.218784,118.120003,149.158188,75.069145
...,...,...,...,...
2026-08-31,511.040009,408.420013,770.739990,97.072998
2026-09-01,500.309998,396.750000,765.349976,96.769997
2026-09-02,501.440002,402.779999,768.809998,96.839996


### 3.2 Data completeness

Measured on `prices_raw`, before `.dropna()` hid anything. A ticker can be short of days for two
very different reasons, so they are counted separately:

- **a late start or early end** — the fund simply did not exist on those dates (`SOXX` and `GLD`
  both predate 2016, but this catches it for any ticker swapped into §2);
- **interior gaps** — a day inside the ticker's own coverage where every other ticker printed a
  price and this one did not. These are the ones worth worrying about.

Any date where even one ticker is missing is dropped from the backtest, so the shared calendar is
the intersection, not the union.

In [11]:
calendar_days = len(prices_raw)          # union calendar: any date at least one ticker traded

gaps_within_coverage =\
(
    prices_raw
    .apply(lambda column: column
                          .loc[column.first_valid_index():column.last_valid_index()]
                          .isna()
                          .sum())      # missing days strictly inside each ticker's own history
)

completeness =\
(
    pd.DataFrame({"First Quote"  : prices_raw.apply(lambda column: column.first_valid_index()),
                  "Last Quote"   : prices_raw.apply(lambda column: column.last_valid_index()),
                  "Trading Days" : calendar_days,
                  "Observed"     : prices_raw.count(),
                  "Missing"      : prices_raw.isna().sum(),
                  "Interior Gaps": gaps_within_coverage})
    .assign(**{"Missing %": lambda frame: (frame["Missing"] / calendar_days).map("{:.2%}".format)})
    .rename_axis("Ticker")
)

print(f"Union calendar : {prices_raw.index[0]:%d %b %Y} -> {prices_raw.index[-1]:%d %b %Y} "
      f"({calendar_days:,} trading days, the union across all tickers)")
print(f"Shared calendar: {len(prices):,} days kept by .dropna() "
      f"({calendar_days - len(prices):,} dropped, {1 - len(prices) / calendar_days:.2%})")

if gaps_within_coverage.any():
    print("Interior gaps (a ticker missing a day the others traded): "
          + ", ".join(f"{ticker} {count}"
                      for ticker, count in gaps_within_coverage[gaps_within_coverage > 0].items()))
else:
    print("No interior gaps: every ticker quotes a price on every day inside its own coverage.")

completeness

Union calendar : 02 Jan 2014 -> 04 Sep 2026 (3,188 trading days, the union across all tickers)
Shared calendar: 3,188 days kept by .dropna() (0 dropped, 0.00%)
No interior gaps: every ticker quotes a price on every day inside its own coverage.


,First Quote,Last Quote,Trading Days,Observed,Missing,Interior Gaps,Missing %
Ticker,,,,,,,
SOXX,2014-01-02,2026-09-04,3188,3188,0,0,0.00%
GLD,2014-01-02,2026-09-04,3188,3188,0,0,0.00%
IVV,2014-01-02,2026-09-04,3188,3188,0,0,0.00%
AGG,2014-01-02,2026-09-04,3188,3188,0,0,0.00%


### 3.3 Risk-free rate

`^TNX` quotes the 10-year Treasury yield as an annualised **percent** (e.g. `2.27` = 2.27%). It is
converted to a compounded daily rate and aligned to the trading calendar of `prices`, so it can be
subtracted from daily returns directly.

In [12]:
rf_daily =\
(
    yf.download(RISK_FREE_TICKER,
                start        = START_DATE,
                end          = END_DATE,
                auto_adjust  = True,
                progress     = False
               )
    ["Close"]
    .squeeze()                          # single-column frame -> Series
    .div(100)                           # percent -> decimal, annualised
    .add(1)
    .pow(1 / TRADING_DAYS)              # annual -> daily, compounded
    .sub(1)
    .reindex(prices.index)              # onto the equity trading calendar
    .ffill()                            # carry the last quote over yield-market holidays
    .bfill()                            # cover a missing first observation
    .rename("rf_daily")
)

rf_daily.to_frame()

,rf_daily
Date,
2014-01-02,0.000068
2014-01-03,0.000068
2014-01-06,0.000067
2014-01-07,0.000066
2014-01-08,0.000069
...,...
2026-08-31,0.000175
2026-09-01,0.000177
2026-09-02,0.000177


### 3.4 Daily asset returns and the derived horizon

Day 0 is inception, so its return is forced to `0` rather than dropped — this keeps every curve
starting at exactly `INITIAL_CAPITAL` on the first date.

In [13]:
asset_returns =\
(
    prices
    .pct_change()                       # daily simple returns
    .fillna(0)                          # day 0 = inception, no return yet
)

HORIZON_DAYS = len(asset_returns)       # horizon, in trading days

print(f"Horizon: {prices.index[0]:%d %b %Y} -> {prices.index[-1]:%d %b %Y} "
      f"({HORIZON_DAYS:,} trading days, {HORIZON_DAYS / TRADING_DAYS:.2f} years)")
print(f"Number of Rebalances over the horizon: {(HORIZON_DAYS - 1) // REBALANCE_DAYS}")

asset_returns

Horizon: 02 Jan 2014 -> 04 Sep 2026 (3,188 trading days, 12.65 years)
Number of Rebalances over the horizon: 53


Ticker,SOXX,GLD,IVV,AGG
Date,,,,
2014-01-02,0.000000,0.000000,0.000000,0.000000
2014-01-03,-0.002928,0.010932,-0.000435,0.000376
2014-01-06,-0.004895,0.001760,-0.002718,0.001314
2014-01-07,0.006887,-0.005690,0.006160,0.000469
2014-01-08,0.014936,-0.005891,0.000542,-0.002998
...,...,...,...,...
2026-08-31,0.004758,-0.001149,-0.002924,-0.000821
2026-09-01,-0.020996,-0.028574,-0.006993,-0.003121
2026-09-02,0.002259,0.015198,0.004521,0.000723


---
## 4. Reusable functions

Five small building blocks, each doing one job: rebalance a portfolio, measure a drawdown,
measure tail risk, summarise risk & return, and regress one excess-return series on another.

In [14]:
w      = pd.Series(INVESTOR_PORTFOLIO, dtype = float)
w
blocks = pd.Series(np.arange(len(asset_returns)) // REBALANCE_DAYS,
                    index = asset_returns.index)

value_per_dollar =\
(
    asset_returns
    [w.index]                        # only this portfolio's assets, in weight order
    .add(1)
    .groupby(blocks)
    # .cumprod()                       # growth of $1 per asset, restarting each block
    # .mul(w, axis = "columns")        # weighted at the block's target weights
    # .sum(axis = "columns")           # -> portfolio value per $1 invested at block start
)
value_per_dollar

In [15]:
def do_rebalance(asset_returns: pd.DataFrame,
                 weights: dict[str, float],
                 initial_capital: float,
                 rebalance_days: int) -> pd.Series:
    """Compound a portfolio daily, snapping back to target `weights` every `rebalance_days`.

    Weights are assumed perfectly divisible: no share counts, no transaction costs.

    Between rebalances each holding drifts with its own return; on a rebalance date the
    portfolio value is redistributed across the target weights and compounding resumes.
    Vectorised by splitting the horizon into fixed-length blocks, so there is no day loop.

    Parameters
    ----------
    asset_returns   : pd.DataFrame      daily simple returns; DatetimeIndex rows x ticker
                                        columns, first row 0.0 (inception)
    weights         : dict[str, float]  target weight per ticker, summing to 1.0; the keys
                                        must be a subset of `asset_returns.columns`
    initial_capital : float             portfolio value on the first date, in dollars
    rebalance_days  : int               trading days between rebalances

    Returns
    -------
    pd.Series  float, named "Value", indexed like `asset_returns`: the portfolio's dollar
               value on each date, starting at `initial_capital`
    """
    w      = pd.Series(weights, dtype = float)
    blocks = pd.Series(np.arange(len(asset_returns)) // rebalance_days,
                       index = asset_returns.index)

    value_per_dollar =\
    (
        asset_returns
        [w.index]                        # only this portfolio's assets, in weight order
        .add(1)
        .groupby(blocks)
        .cumprod()                       # growth of $1 per asset, restarting each block
        .mul(w, axis = "columns")        # weighted at the block's target weights
        .sum(axis = "columns")           # -> portfolio value per $1 invested at block start
    )

    capital_at_block_start =\
    (
        value_per_dollar
        .groupby(blocks)
        .last()                          # each block's total growth factor
        .shift(1, fill_value = 1.0)      # block N starts with what blocks 0..N-1 earned
        .cumprod()
        .mul(initial_capital)
    )

    return (
        value_per_dollar
        .mul(blocks.map(capital_at_block_start))
        .rename("Value")
    )

In [16]:
def compute_drawdown(value: pd.Series) -> pd.Series:
    """Percentage decline from the running peak, for every date.

    Parameters
    ----------
    value : pd.Series  a positive level series, e.g. the dollar values from `do_rebalance`

    Returns
    -------
    pd.Series  float, indexed like `value`: 0.0 on a day that sets a new peak, negative
               below it (-0.25 = 25% under water)
    """
    return (
        value
        .div(value.cummax())
        .sub(1)
    )


def compute_cvar(portfolio_returns: pd.Series,
                 confidence: float = 0.95) -> float:
    """Conditional value at risk: the average daily return on the worst `1 - confidence` days.

    Historical (non-parametric): no distribution is assumed, the sample's own left tail *is*
    the estimate. Value at risk is the quantile that cuts the tail off; CVaR (expected
    shortfall) is the mean of everything beyond it, so it answers "when a bad day happens,
    how bad on average" rather than "how bad at the threshold" — and unlike VaR it is
    sensitive to how fat the tail is behind the cutoff.

    Parameters
    ----------
    portfolio_returns : pd.Series  daily simple returns of one portfolio
    confidence        : float      tail cutoff, 0.95 -> the worst 5% of days

    Returns
    -------
    float  negative decimal, same sign convention as a drawdown (-0.03 = the worst 5% of
           days lose 3% on average)
    """
    value_at_risk = portfolio_returns.quantile(1 - confidence)   # left-tail cutoff

    return (
        portfolio_returns
        [portfolio_returns <= value_at_risk]                     # the tail itself
        .mean()
    )

In [17]:
def compute_metrics(portfolio_returns: pd.Series,
                    rf_daily: pd.Series,
                    trading_days: int) -> dict[str, float]:
    """Return / volatility / Sharpe, reported both over the full horizon and annualised.

    Parameters
    ----------
    portfolio_returns : pd.Series  daily simple returns of one portfolio
    rf_daily          : pd.Series  daily risk-free rate, sharing the same index
    trading_days      : int        annualisation factor (252)

    Returns
    -------
    dict[str, float]  six metrics keyed by name: "Horizon Return", "Annualised Return",
                      "Horizon Volatility", "Annualised Volatility", "Horizon Sharpe",
                      "Annualised Sharpe". Returns and volatilities are decimals
                      (0.12 = 12%); the Sharpe ratios are unitless.
    """
    horizon_days   = len(portfolio_returns)
    horizon_return = portfolio_returns.add(1).prod() - 1
    daily_vol      = portfolio_returns.std()

    excess_returns = portfolio_returns.sub(rf_daily)
    annual_sharpe  = excess_returns.mean() / excess_returns.std() * np.sqrt(trading_days)

    return {
        "Horizon Return"       : horizon_return,
        "Annualised Return"    : (1 + horizon_return) ** (trading_days / horizon_days) - 1,
        "Horizon Volatility"   : daily_vol * np.sqrt(horizon_days),
        "Annualised Volatility": daily_vol * np.sqrt(trading_days),
        "Horizon Sharpe"       : annual_sharpe * np.sqrt(horizon_days / trading_days),
        "Annualised Sharpe"    : annual_sharpe,
    }

In [18]:
def compute_alpha_beta(excess_investor: pd.Series,
                       excess_benchmark: pd.Series,
                       trading_days: int,
                       confidence: float = 0.95) -> dict[str, float]:
    """OLS of the investor's excess return on the benchmark's excess return, with inference.

        R_p - R_f = alpha + beta * (R_m - R_f) + epsilon

    beta  -> sensitivity to the benchmark (just the market)
    alpha -> average excess return the benchmark does not explain (skill)

    Tests H_0: alpha = 0 two ways. The textbook standard error assumes i.i.d. homoskedastic
    residuals; daily returns are neither, being volatility-clustered and mildly autocorrelated,
    so a Newey-West HAC standard error is reported alongside it. Where the two disagree, trust
    the HAC one -- the textbook error is the optimistic one.

    Parameters
    ----------
    excess_investor  : pd.Series  daily investor excess returns  (R_p - R_f), the regressand
    excess_benchmark : pd.Series  daily benchmark excess returns (R_m - R_f), the regressor;
                                  same index as `excess_investor`
    trading_days     : int        annualisation factor (252)
    confidence       : float      two-sided confidence level for the alpha interval (0.95)

    Returns
    -------
    dict[str, float]  "Beta" (unitless slope), "Alpha (daily)" (decimal intercept),
                      "Annualised Alpha" (decimal, alpha compounded over `trading_days`),
                      "R-squared" (0.0 to 1.0), then the inference on alpha:
                      "SE Alpha (OLS)" / "t-stat (OLS)" / "p-value (OLS)" under i.i.d. errors,
                      "SE Alpha (HAC)" / "t-stat (HAC)" / "p-value (HAC)" Newey-West corrected,
                      "NW Lags" (Bartlett truncation lag), and the annualised alpha interval
                      "Annual Alpha CI Low" / "Annual Alpha CI High" (decimals, from the HAC SE)
    """
    beta        = excess_investor.cov(excess_benchmark) / excess_benchmark.var()
    alpha_daily = excess_investor.mean() - beta * excess_benchmark.mean()

    # --- residuals and the textbook (i.i.d.) standard error of the intercept ---
    residuals = excess_investor - alpha_daily - beta * excess_benchmark
    n         = len(excess_investor)
    dof       = n - 2                                    # two estimated parameters
    sigma2    = (residuals ** 2).sum() / dof
    x_mean    = excess_benchmark.mean()
    sxx       = ((excess_benchmark - x_mean) ** 2).sum()

    se_ols    = np.sqrt(sigma2 * (1 / n + x_mean ** 2 / sxx))

    # --- Newey-West HAC standard error: sandwich with Bartlett-weighted autocovariances ---
    design    = np.column_stack([np.ones(n), excess_benchmark.to_numpy()])
    scores    = residuals.to_numpy()[:, None] * design   # h_t = e_t * X_t
    nw_lags   = int(4 * (n / 100) ** (2 / 9))            # standard truncation rule

    meat = scores.T @ scores
    for lag in range(1, nw_lags + 1):
        gamma = scores[lag:].T @ scores[:-lag]
        meat += (1 - lag / (nw_lags + 1)) * (gamma + gamma.T)

    bread     = np.linalg.inv(design.T @ design)
    se_hac    = np.sqrt((bread @ meat @ bread)[0, 0])    # [0, 0] is the intercept's variance

    t_ols     = alpha_daily / se_ols
    t_hac     = alpha_daily / se_hac
    t_crit    = stats.t.ppf(0.5 + confidence / 2, dof)

    # Compounding is monotone, so transforming the daily endpoints preserves the coverage.
    ci_daily  = alpha_daily - t_crit * se_hac, alpha_daily + t_crit * se_hac

    return {
        "Beta"                : beta,
        "Alpha (daily)"       : alpha_daily,
        "Annualised Alpha"    : (1 + alpha_daily) ** trading_days - 1,
        "R-squared"           : excess_investor.corr(excess_benchmark) ** 2,
        "SE Alpha (OLS)"      : se_ols,
        "t-stat (OLS)"        : t_ols,
        "p-value (OLS)"       : 2 * stats.t.sf(abs(t_ols), dof),
        "SE Alpha (HAC)"      : se_hac,
        "t-stat (HAC)"        : t_hac,
        "p-value (HAC)"       : 2 * stats.t.sf(abs(t_hac), dof),
        "NW Lags"             : nw_lags,
        "Annual Alpha CI Low" : (1 + ci_daily[0]) ** trading_days - 1,
        "Annual Alpha CI High": (1 + ci_daily[1]) ** trading_days - 1,
    }

---
## 5. Backtest

### 5.1 Portfolio values

Both portfolios start at `INITIAL_CAPITAL` and compound daily, rebalancing every
`REBALANCE_DAYS`.

In [19]:
portfolio_values =\
(
    pd.DataFrame(
        {"Investor" : do_rebalance(asset_returns, INVESTOR_PORTFOLIO,
                                   INITIAL_CAPITAL, REBALANCE_DAYS),
         "Benchmark": do_rebalance(asset_returns, BENCHMARK_PORTFOLIO,
                                   INITIAL_CAPITAL, REBALANCE_DAYS)}
    )
    .rename_axis("Date")
)

portfolio_values

,Investor,Benchmark
Date,,
2014-01-02,"100,000.000000","100,000.000000"
2014-01-03,"100,122.989025","99,988.926295"
2014-01-06,"99,834.715747","99,878.489469"
2014-01-07,"100,140.131715","100,265.720816"
2014-01-08,"101,006.671501","100,178.158791"
...,...,...
2026-08-31,"1,538,324.531096","306,517.910536"
2026-09-01,"1,502,628.158882","304,846.903050"
2026-09-02,"1,511,657.586377","305,763.035139"


### 5.2 Portfolio returns and excess returns

Rebalancing is value-neutral (it only reshuffles an unchanged total), so the daily return of the
portfolio is simply the day-over-day change in its value, including across rebalance dates.

In [20]:
portfolio_returns =\
(
    portfolio_values
    .pct_change()
    .fillna(0)                           # day 0 = inception
)

portfolio_returns

,Investor,Benchmark
Date,,
2014-01-02,0.000000,0.000000
2014-01-03,0.001230,-0.000111
2014-01-06,-0.002879,-0.001104
2014-01-07,0.003059,0.003877
2014-01-08,0.008653,-0.000873
...,...,...
2026-08-31,0.003029,-0.002087
2026-09-01,-0.023205,-0.005452
2026-09-02,0.006009,0.003005


In [21]:
excess_returns =\
(
    portfolio_returns
    .sub(rf_daily, axis = "index")       # R - R_f, the input to Sharpe and to alpha/beta
)

excess_returns

,Investor,Benchmark
Date,,
2014-01-02,-0.000068,-0.000068
2014-01-03,0.001162,-0.000179
2014-01-06,-0.002946,-0.001171
2014-01-07,0.002993,0.003811
2014-01-08,0.008584,-0.000943
...,...,...
2026-08-31,0.002854,-0.002262
2026-09-01,-0.023382,-0.005628
2026-09-02,0.005832,0.002829


---
## 6. Result analysis

### 6.1 Return, volatility and Sharpe

In [22]:
metrics_table =\
(
    pd.DataFrame(
        {column: compute_metrics(portfolio_returns[column], rf_daily, TRADING_DAYS)
         for column in portfolio_values.columns}
    )
    .rename_axis("Metric")
)

metrics_table

,Investor,Benchmark
Metric,,
Horizon Return,14.553253,2.071872
Annualised Return,0.242251,0.092767
Horizon Volatility,0.835288,0.376121
Annualised Volatility,0.234843,0.105747
Horizon Sharpe,3.347738,2.380368
Annualised Sharpe,0.941223,0.669245


### 6.2 Drawdowns and tail risk

Two views of downside: the **max drawdown** is the worst peak-to-trough loss the path ever sustained, a
peak-relative and path-dependent number; **CVaR** is the average loss on the worst
$1 - \text{confidence}$ of individual days, which ignores the path but says how heavy the left tail is.
One bounds the pain of holding on, the other the pain of a single bad day.

CVaR is reported twice: as a percentage, and as that percentage applied to the portfolio's **latest**
value. The dollar figure is what a worst-5% day costs on the balance actually held today, so it grows
with the account rather than staying pinned to `INITIAL_CAPITAL`.

In [23]:
drawdowns =\
(
    portfolio_values
    .apply(compute_drawdown)
)

CVAR_LABEL        = f"CVaR ({CVAR_CONFIDENCE:.0%}, daily)"
CVAR_DOLLAR_LABEL = f"CVaR ({CVAR_CONFIDENCE:.0%}, daily $)"

latest_value =\
(
    portfolio_values
    .iloc[-1]                            # a Series per portfolio, so it aligns on the index
)

cvar =\
(
    portfolio_returns
    .apply(compute_cvar, confidence = CVAR_CONFIDENCE)
)

risk_table =\
(
    pd.DataFrame({"Max Drawdown"     : drawdowns.min(),
                  "Max Drawdown Date": drawdowns.idxmin(),
                  CVAR_LABEL         : cvar,
                  CVAR_DOLLAR_LABEL  : cvar.mul(latest_value)})   # same loss, on today's balance
    .rename_axis("Portfolio")
)

print("Investor portfolio worst loss from a prior peak: "
      f"{drawdowns['Investor'].min():.2%} on {drawdowns['Investor'].idxmin():%d %b %Y}")
print(f"Investor portfolio {CVAR_CONFIDENCE:.0%} CVaR: "
      f"{risk_table.loc['Investor', CVAR_LABEL]:.2%} — the average return on its worst "
      f"{(1 - CVAR_CONFIDENCE) * len(portfolio_returns):.0f} days, "
      f"or -${abs(risk_table.loc['Investor', CVAR_DOLLAR_LABEL]):,.0f} "
      f"on its latest value of ${latest_value['Investor']:,.0f}")

risk_table

Investor portfolio worst loss from a prior peak: -36.38% on 14 Oct 2022
Investor portfolio 95% CVaR: -3.44% — the average return on its worst 159 days, or -$53,456 on its latest value of $1,555,325


,Max Drawdown,Max Drawdown Date,"CVaR (95%, daily)","CVaR (95%, daily $)"
Portfolio,,,,
Investor,-0.363821,2022-10-14,-0.034370,"-53,455.756312"
Benchmark,-0.218240,2020-03-23,-0.015855,"-4,870.398258"


### 6.3 Alpha and beta

Investor excess return regressed on benchmark excess return.

The intercept is only interesting if it is distinguishable from zero, so the regression also tests

$$H_0:\ \alpha = 0 \qquad \text{against} \qquad H_1:\ \alpha \neq 0$$

Two standard errors are reported. The **OLS** one is the textbook formula, which assumes the
residuals are independent and identically distributed. Daily returns are neither — they are
volatility-clustered and mildly autocorrelated — so that error is typically too small and the
significance it implies too generous. The **Newey-West (HAC)** standard error corrects for both
heteroskedasticity and autocorrelation up to a truncation lag. Where the two disagree, the HAC
verdict is the one to believe.

In [24]:
# === 1 day rebalancing ===
# R_p - R_f = 0.000199 + 2.0184 * (R_m - R_f)      [daily, R^2 = 0.8309]

# H_0: alpha = 0     (n = 1,425 daily observations)
#   OLS         SE = 0.000162   t =  1.226   p = 0.2202
#   Newey-West  SE = 0.000154   t =  1.298   p = 0.1946   [7 lags]

# At the 5% level we cannot reject H_0 on the HAC standard error.
# Annualised alpha 5.15%, 95% CI [-2.54%, 13.43%]


# === 60 day rebalancing ===
# R_p - R_f = 0.000203 + 2.0278 * (R_m - R_f)      [daily, R^2 = 0.8309]

# H_0: alpha = 0     (n = 1,425 daily observations)
#   OLS         SE = 0.000162   t =  1.248   p = 0.2122
#   Newey-West  SE = 0.000153   t =  1.324   p = 0.1858   [7 lags]

# At the 5% level we cannot reject H_0 on the HAC standard error.
# Annualised alpha 5.24%, 95% CI [-2.43%, 13.52%]


# === 100 day rebalancing ===
# R_p - R_f = 0.000209 + 2.0219 * (R_m - R_f)      [daily, R^2 = 0.8320]

# H_0: alpha = 0     (n = 1,425 daily observations)
#   OLS         SE = 0.000162   t =  1.292   p = 0.1967
#   Newey-West  SE = 0.000153   t =  1.365   p = 0.1726   [7 lags]

# At the 5% level we cannot reject H_0 on the HAC standard error.
# Annualised alpha 5.41%, 95% CI [-2.28%, 13.71%]



# === Without Rebalancing ===
# R_p - R_f = 0.000157 + 1.8483 * (R_m - R_f)      [daily, R^2 = 0.8523]

# H_0: alpha = 0     (n = 1,425 daily observations)
#   OLS         SE = 0.000152   t =  1.028   p = 0.3043
#   Newey-West  SE = 0.000145   t =  1.083   p = 0.2790   [7 lags]

# At the 5% level we cannot reject H_0 on the HAC standard error.
# Annualised alpha 4.03%, 95% CI [-3.15%, 11.73%]

In [25]:
alpha_beta = compute_alpha_beta(excess_returns["Investor"],
                                excess_returns["Benchmark"],
                                TRADING_DAYS)

print(f"R_p - R_f = {alpha_beta['Alpha (daily)']:.6f} "
      f"+ {alpha_beta['Beta']:.4f} * (R_m - R_f)      [daily, R^2 = {alpha_beta['R-squared']:.4f}]")

print(f"\nH_0: alpha = 0     (n = {len(excess_returns):,} daily observations)")
print(f"  OLS         SE = {alpha_beta['SE Alpha (OLS)']:.6f}   "
      f"t = {alpha_beta['t-stat (OLS)']:6.3f}   p = {alpha_beta['p-value (OLS)']:.4f}")
print(f"  Newey-West  SE = {alpha_beta['SE Alpha (HAC)']:.6f}   "
      f"t = {alpha_beta['t-stat (HAC)']:6.3f}   p = {alpha_beta['p-value (HAC)']:.4f}   "
      f"[{alpha_beta['NW Lags']} lags]")

verdict = "reject" if alpha_beta["p-value (HAC)"] < 0.05 else "cannot reject"
print(f"\nAt the 5% level we {verdict} H_0 on the HAC standard error.")
print(f"Annualised alpha {alpha_beta['Annualised Alpha']:.2%}, "
      f"95% CI [{alpha_beta['Annual Alpha CI Low']:.2%}, "
      f"{alpha_beta['Annual Alpha CI High']:.2%}]")

pd.Series(alpha_beta).rename("Investor vs Benchmark").to_frame()

R_p - R_f = 0.000397 + 1.7081 * (R_m - R_f)      [daily, R^2 = 0.5917]

H_0: alpha = 0     (n = 3,188 daily observations)
  OLS         SE = 0.000168   t =  2.371   p = 0.0178
  Newey-West  SE = 0.000149   t =  2.662   p = 0.0078   [8 lags]

At the 5% level we reject H_0 on the HAC standard error.
Annualised alpha 10.53%, 95% CI [2.67%, 18.99%]


,Investor vs Benchmark
Beta,1.708121
Alpha (daily),0.000397
Annualised Alpha,0.105308
R-squared,0.591689
SE Alpha (OLS),0.000168
...,...
t-stat (HAC),2.662140
p-value (HAC),0.007804
NW Lags,8.000000
Annual Alpha CI Low,0.026736


### 6.4 Summary table

Every computed figure in one place: returns, volatility, Sharpe, max drawdown and its date, CVaR,
then the whole regression — alpha and beta, both standard errors and their t-statistics and
p-values, the Newey-West truncation lag, and the annualised alpha confidence interval.

Alpha and beta describe the investor *relative to* the benchmark, so they are blank in the
benchmark's own column.

In [26]:
summary_table =\
(
    metrics_table
    .T                                                       # portfolios -> rows
    # every risk column, then every key the regression returned; the regression rows are
    # investor-only, because the benchmark cannot have an alpha or a beta against itself
    .assign(**{metric: risk_table[metric] for metric in risk_table.columns},
            **{metric: [value, np.nan] for metric, value in alpha_beta.items()})
    .T                                                       # metrics -> rows
    .rename_axis("Metric")
)

summary_table

,Investor,Benchmark
Metric,,
Horizon Return,14.553253,2.071872
Annualised Return,0.242251,0.092767
Horizon Volatility,0.835288,0.376121
Annualised Volatility,0.234843,0.105747
Horizon Sharpe,3.347738,2.380368
...,...,...
t-stat (HAC),2.662140,NaN
p-value (HAC),0.007804,NaN
NW Lags,8.000000,NaN


In [27]:
# Same table, formatted for reading. Each metric carries its own convention: rates as
# percentages, dollar losses with a sign in front of the currency symbol, daily alpha and its
# standard errors too small for a percentage, p-values that would round to a bare 0.0000, and
# the lag count as a plain integer.
percent_metrics = ["Horizon Return", "Annualised Return", "Horizon Volatility",
                   "Annualised Volatility", "Max Drawdown", CVAR_LABEL, "Annualised Alpha",
                   "Annual Alpha CI Low", "Annual Alpha CI High"]
dollar_metrics  = [CVAR_DOLLAR_LABEL]
daily_metrics   = ["Alpha (daily)", "SE Alpha (OLS)", "SE Alpha (HAC)"]
pvalue_metrics  = ["p-value (OLS)", "p-value (HAC)"]
count_metrics   = ["NW Lags"]


def format_metric(value: float, metric: str) -> str:
    """Render one cell of `summary_table`, choosing the convention from its metric name.

    Parameters
    ----------
    value  : float  the cell's value; `NaN` where the metric does not apply to that portfolio
    metric : str    the row label, i.e. which convention to use

    Returns
    -------
    str  the display string, "—" for a metric that does not apply
    """
    if pd.isna(value):
        return "—"

    if metric in percent_metrics:
        return f"{value:.2%}"
    if metric in dollar_metrics:
        return f"-${abs(value):,.0f}" if value < 0 else f"${value:,.0f}"
    if metric == "Max Drawdown Date":
        return f"{value:%d %b %Y}"
    if metric in daily_metrics:
        return f"{value:.6f}"
    if metric in pvalue_metrics:
        return "<0.0001" if value < 1e-4 else f"{value:.4f}"
    if metric in count_metrics:
        return f"{value:,.0f}"

    return f"{value:.4f}"


summary_display =\
(
    summary_table
    .apply(lambda row: row.map(lambda value: format_metric(value, row.name)),
           axis = "columns")
)

summary_display

,Investor,Benchmark
Metric,,
Horizon Return,1455.33%,207.19%
Annualised Return,24.23%,9.28%
Horizon Volatility,83.53%,37.61%
Annualised Volatility,23.48%,10.57%
Horizon Sharpe,3.3477,2.3804
...,...,...
t-stat (HAC),2.6621,—
p-value (HAC),0.0078,—
NW Lags,8,—


---
## 7. Charts

In [28]:
portfolio_values_long =\
(
    portfolio_values
    .reset_index()
    .melt(id_vars    = "Date",
          var_name   = "Portfolio",
          value_name = "Value")
)

portfolio_values_long

,Date,Portfolio,Value
0,2014-01-02,Investor,"100,000.000000"
1,2014-01-03,Investor,"100,122.989025"
2,2014-01-06,Investor,"99,834.715747"
3,2014-01-07,Investor,"100,140.131715"
4,2014-01-08,Investor,"101,006.671501"
...,...,...,...
6371,2026-08-31,Benchmark,"306,517.910536"
6372,2026-09-01,Benchmark,"304,846.903050"
6373,2026-09-02,Benchmark,"305,763.035139"
6374,2026-09-03,Benchmark,"307,892.649609"


In [29]:
def describe(portfolio: dict[str, float]) -> str:
    """A weight dict as a one-line label, e.g. "SOXX 70% / GLD 30%".

    Parameters
    ----------
    portfolio : dict[str, float]  ticker -> target weight, as declared in §2

    Returns
    -------
    str  the holdings joined with " / ", each weight as a whole percentage
    """
    return " / ".join(f"{ticker} {weight:.0%}" for ticker, weight in portfolio.items())


# Labels are derived from §2, not typed out, so they cannot drift from the weights actually
# backtested. lets-plot orders a discrete scale alphabetically: Benchmark first, then Investor.
equity_plot =\
(
    ggplot(portfolio_values_long,
           aes(x = "Date",
               y = "Value")
          )
    + geom_line(aes(color = "Portfolio"),
                size = 0.8)
    + scale_color_manual(values = ["blue", "red"],
                         labels = [f"Benchmark: {describe(BENCHMARK_PORTFOLIO)}",
                                   f"Investor: {describe(INVESTOR_PORTFOLIO)}"],
                         name   = "Portfolio")
    + scale_y_continuous(format = "$,.0f")
    + labs(title    = f"Growth of ${INITIAL_CAPITAL:,.0f}, rebalanced every {REBALANCE_DAYS} trading days",
           subtitle = f"{prices.index[0]:%d %b %Y} to {prices.index[-1]:%d %b %Y}",
           x        = "",
           y        = "Portfolio Value")
    + ggsize(1000, 500)
    + theme(legend_position = "top")
)

equity_plot

In [30]:
drawdown_plot =\
(
    ggplot(drawdowns.reset_index()
                    .melt(id_vars    = "Date",
                          var_name   = "Portfolio",
                          value_name = "Drawdown"),
           aes(x = "Date",
               y = "Drawdown")
          )
    + geom_area(aes(fill = "Portfolio"),
                alpha = 0.35)
    + geom_line(aes(color = "Portfolio"),
                size = 0.5)
    + scale_fill_manual(values  = ["blue", "red"], name = "Portfolio")
    + scale_color_manual(values = ["blue", "red"], name = "Portfolio")
    + scale_y_continuous(format = ".0%")
    + labs(title = "Underwater plot: decline from the running peak",
           x     = "",
           y     = "Drawdown")
    + ggsize(1000, 400)
    + theme(legend_position = "top")
)

drawdown_plot

In [31]:
# The regression behind alpha and beta, drawn: each point is one trading day.
alpha_beta_plot =\
(
    ggplot(excess_returns.reset_index(),
           aes(x = "Benchmark",
               y = "Investor")
          )
    + geom_point(color = "grey",
                 alpha = 0.20,
                 size  = 1.5)
    + geom_abline(slope     = alpha_beta["Beta"],
                  intercept = alpha_beta["Alpha (daily)"],
                  color     = "red",
                  size      = 1.0)
    + geom_hline(yintercept = 0, color = "black", size = 0.3)
    + geom_vline(xintercept = 0, color = "black", size = 0.3)
    + scale_x_continuous(format = ".1%")
    + scale_y_continuous(format = ".1%")
    + labs(title    = (f"R_p - R_f  =  {alpha_beta['Alpha (daily)']:.6f}  "
                       f"+  {alpha_beta['Beta']:.4f} (R_m - R_f)"),
           subtitle = (f"Daily excess returns  |  R-squared = {alpha_beta['R-squared']:.4f}  |  "
                       f"annualised alpha = {alpha_beta['Annualised Alpha']:.2%}  |  "
                       f"Newey-West t = {alpha_beta['t-stat (HAC)']:.2f}, p = {alpha_beta['p-value (HAC)']:.4f}"),
           x        = "Benchmark excess return  (R_m - R_f)",
           y        = "Investor excess return  (R_p - R_f)")
    + ggsize(700, 700)
)

alpha_beta_plot

In [32]:
dashboard =\
(
    gggrid([equity_plot, drawdown_plot],
           ncol = 1)
    + ggsize(1000, 900)
)

dashboard

## 8. At a glance

In [37]:
# The whole result in one output. Change the weights in §2, run all, then read only this cell:
# every figure and label below is derived, so nothing here ever needs editing.
summary_charts =\
(
    gggrid([equity_plot, drawdown_plot],
           ncol = 1)
    + ggsize(950, 520)                   # compact enough to sit above the table; §7 keeps the big one
)

verdict = "reject" if alpha_beta["p-value (HAC)"] < 0.05 else "cannot reject"

# Dollar signs are escaped last, so MathJax cannot read the pair spanning the CVaR row as maths.
summary_markdown =\
(
    f"""
---

### **Investor** {describe(INVESTOR_PORTFOLIO)} &nbsp; vs &nbsp; **Benchmark** {describe(BENCHMARK_PORTFOLIO)}

{prices.index[0]:%d %b %Y} → {prices.index[-1]:%d %b %Y} · {HORIZON_DAYS:,} trading days · ${INITIAL_CAPITAL:,.0f} initial · rebalanced every {REBALANCE_DAYS} trading days

{summary_display.to_markdown()}

At the 5% level we **{verdict}** H₀: α = 0 on the Newey-West standard error (p = {alpha_beta['p-value (HAC)']:.4f}).
"""
    .replace("$", r"\$")
)

display(Markdown(summary_markdown))
display(dashboard)


---

### **Investor** SOXX 70% / GLD 30% &nbsp; vs &nbsp; **Benchmark** IVV 60% / AGG 40%

02 Jan 2014 → 04 Sep 2026 · 3,188 trading days · \$100,000 initial · rebalanced every 60 trading days

| Metric                | Investor    | Benchmark   |
|:----------------------|:------------|:------------|
| Horizon Return        | 1455.33%    | 207.19%     |
| Annualised Return     | 24.23%      | 9.28%       |
| Horizon Volatility    | 83.53%      | 37.61%      |
| Annualised Volatility | 23.48%      | 10.57%      |
| Horizon Sharpe        | 3.3477      | 2.3804      |
| Annualised Sharpe     | 0.9412      | 0.6692      |
| Max Drawdown          | -36.38%     | -21.82%     |
| Max Drawdown Date     | 14 Oct 2022 | 23 Mar 2020 |
| CVaR (95%, daily)     | -3.44%      | -1.59%      |
| CVaR (95%, daily \$)   | -\$53,456    | -\$4,870     |
| Beta                  | 1.7081      | —           |
| Alpha (daily)         | 0.000397    | —           |
| Annualised Alpha      | 10.53%      | —           |
| R-squared             | 0.5917      | —           |
| SE Alpha (OLS)        | 0.000168    | —           |
| t-stat (OLS)          | 2.3711      | —           |
| p-value (OLS)         | 0.0178      | —           |
| SE Alpha (HAC)        | 0.000149    | —           |
| t-stat (HAC)          | 2.6621      | —           |
| p-value (HAC)         | 0.0078      | —           |
| NW Lags               | 8           | —           |
| Annual Alpha CI Low   | 2.67%       | —           |
| Annual Alpha CI High  | 18.99%      | —           |

At the 5% level we **reject** H₀: α = 0 on the Newey-West standard error (p = 0.0078).
